# 07 — Cross-Model Comparison: Qwen2.5-7B-Instruct

Replicates Phase 1 (contrastive feature identification) and Phase 2b (causal H1 ablation) on
**Qwen2.5-7B-Instruct** with BatchTopK SAEs from [andyrdt/saes-qwen2.5-7b-instruct](https://huggingface.co/andyrdt/saes-qwen2.5-7b-instruct).

Goal: test whether language-reasoning interference generalizes across model families (Gemma 2 → Qwen 2.5).

**Run on Colab Pro+ A100 or Prime Intellect A100 80GB.**

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

# --- Environment detection ---
IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/kvrancic/nlp-project.git'
REPO_DIR = '/content/nlp-project' if IN_COLAB else Path.cwd().parent

if IN_COLAB:
    if not Path(REPO_DIR).exists():
        !git clone {REPO_URL} {REPO_DIR}
    else:
        !cd {REPO_DIR} && git pull --ff-only
    os.chdir(REPO_DIR)
    !pip install -q 'numpy>=2.0' -e .
    !pip install -q git+https://github.com/saprmarks/dictionary_learning.git
else:
    os.chdir(str(REPO_DIR))

# Bust cached src modules
for _mod_name in [m for m in list(sys.modules) if m == 'src' or m.startswith('src.')]:
    del sys.modules[_mod_name]

# HF token
if IN_COLAB:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    Path('.env').write_text(f'HF_TOKEN={os.environ["HF_TOKEN"]}\n')
else:
    from dotenv import load_dotenv
    load_dotenv()

# Drive mount for persistent results
DRIVE_RESULTS = None
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DRIVE_RESULTS = Path('/content/drive/MyDrive/nlp-project-results')
        DRIVE_RESULTS.mkdir(exist_ok=True, parents=True)
        print(f'Drive results dir: {DRIVE_RESULTS}')
    except Exception as e:
        print(f'Drive mount failed ({e})')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

from src.config import (
    TARGET_LANGUAGES, RESULTS_DIR,
    QWEN_MODEL_ID, QWEN_D_MODEL, QWEN_N_LAYERS,
    QWEN_SAE_SUBSET_LAYERS, QWEN_SAE_WIDTH, QWEN_SAE_TRAINER,
)
from src.data import load_mgsm
from src.model import load_model_and_tokenizer, load_qwen_saes_at_layers
from src.extraction import extract_residual_activations, encode_activations_batchtopk
from src.monolinguality import (
    compute_monolinguality, identify_language_features,
    identify_reasoning_features, train_language_probe, probe_language_features,
)
from src.intervention import get_sae_decoder_directions, run_generate_with_hooks

torch.manual_seed(0)
np.random.seed(0)

print(f'Model: {QWEN_MODEL_ID}')
print(f'SAE layers: {QWEN_SAE_SUBSET_LAYERS}')
print(f'Languages: {TARGET_LANGUAGES}')

## 1. Load Qwen2.5-7B-Instruct + SAEs

In [ ]:
model, tokenizer = load_model_and_tokenizer(model_id=QWEN_MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print('Set tokenizer.pad_token = eos_token')

print(f'Model loaded: {QWEN_MODEL_ID}')
print(f'  d_model={QWEN_D_MODEL}, n_layers={QWEN_N_LAYERS}')

In [ ]:
saes = load_qwen_saes_at_layers(
    layers=QWEN_SAE_SUBSET_LAYERS,
    trainer=QWEN_SAE_TRAINER,
)
for layer, ae in saes.items():
    n_params = sum(p.numel() for p in ae.parameters())
    print(f'  layer {layer:>2}: {n_params/1e6:.1f}M params')

## 2. Load MGSM and build prompts

In [ ]:
mgsm = load_mgsm(TARGET_LANGUAGES)
for lang in TARGET_LANGUAGES:
    print(f'  {lang}: {len(mgsm[lang])} problems')

def make_prompt(question: str) -> str:
    return tokenizer.apply_chat_template(
        [{'role': 'user', 'content': question}],
        tokenize=False,
        add_generation_prompt=True,
    )

prompts_by_lang = {
    lang: [make_prompt(ex['question']) for ex in mgsm[lang]]
    for lang in TARGET_LANGUAGES
}
for lang in TARGET_LANGUAGES:
    print(f'  {lang}: {len(prompts_by_lang[lang])} prompts')

## 3. Extract residual-stream activations

In [ ]:
activations = {layer: {} for layer in QWEN_SAE_SUBSET_LAYERS}

for lang in TARGET_LANGUAGES:
    print(f'\n=== {lang} ===')
    acts = extract_residual_activations(
        model, tokenizer, prompts_by_lang[lang],
        layers=QWEN_SAE_SUBSET_LAYERS,
        batch_size=4,
        positions='last',
    )
    for layer in QWEN_SAE_SUBSET_LAYERS:
        activations[layer][lang] = acts[layer]
        print(f'  layer {layer:>2}: shape={tuple(acts[layer].shape)}, '
              f'norm_mean={acts[layer].float().norm(dim=-1).mean():.2f}')
    torch.cuda.empty_cache()

for layer in QWEN_SAE_SUBSET_LAYERS:
    assert all(activations[layer][l].shape == (250, QWEN_D_MODEL) for l in TARGET_LANGUAGES)
print('\nAll activation shapes (250, 3584) verified.')

## 4. Encode through BatchTopK SAEs

In [ ]:
feature_acts = {layer: {} for layer in QWEN_SAE_SUBSET_LAYERS}

for layer in QWEN_SAE_SUBSET_LAYERS:
    ae = saes[layer]
    for lang in TARGET_LANGUAGES:
        feats = encode_activations_batchtopk(
            activations[layer][lang], ae, batch_size=64,
        )
        feature_acts[layer][lang] = feats  # (250, 131072)
    sample = feature_acts[layer][TARGET_LANGUAGES[0]]
    n_active_per_tok = (sample > 0).float().sum(dim=-1).mean().item()
    print(f'  layer {layer:>2}: feats {tuple(sample.shape)}, '
          f'mean active features/example = {n_active_per_tok:.1f}')

torch.cuda.empty_cache()

## 5. Method A — Monolinguality metric

In [ ]:
TOP_K = 50
monolinguality_per_layer = {}
top_features_A = {layer: {} for layer in QWEN_SAE_SUBSET_LAYERS}

for layer in QWEN_SAE_SUBSET_LAYERS:
    mono = compute_monolinguality(feature_acts[layer])
    monolinguality_per_layer[layer] = mono
    top_features_A[layer] = identify_language_features(mono, top_k=TOP_K)
    print(f'  layer {layer:>2}:')
    for lang in TARGET_LANGUAGES:
        top1 = top_features_A[layer][lang][0]
        score = mono[lang][top1].item()
        print(f'    {lang}: top1 feature={top1}, \u03bd={score:.4f}')

## 6. Method B — Supervised language probe

In [ ]:
probe_per_layer = {}
top_features_B = {layer: {} for layer in QWEN_SAE_SUBSET_LAYERS}
probe_accuracies = {}

for layer in QWEN_SAE_SUBSET_LAYERS:
    clf, importances = train_language_probe(feature_acts[layer], max_iter=2000)
    probe_per_layer[layer] = (clf, importances)
    top_features_B[layer] = probe_language_features(
        clf, importances, sorted(TARGET_LANGUAGES), top_k=TOP_K,
    )
    X = np.concatenate([feature_acts[layer][l].float().numpy() for l in sorted(TARGET_LANGUAGES)], axis=0)
    y = np.concatenate([np.full(250, i) for i in range(len(TARGET_LANGUAGES))], axis=0)
    acc = clf.score(X, y)
    probe_accuracies[layer] = acc
    print(f'  layer {layer:>2}: probe in-sample accuracy = {acc:.3f}')

## 7. Method C — Cross-lingual reasoning features

In [ ]:
reasoning_features_per_layer = {}
for layer in QWEN_SAE_SUBSET_LAYERS:
    reason = identify_reasoning_features(feature_acts[layer], threshold=0.1)
    reasoning_features_per_layer[layer] = reason
    print(f'  layer {layer:>2}: {len(reason)} cross-lingual reasoning candidates '
          f'(out of {feature_acts[layer][TARGET_LANGUAGES[0]].shape[1]})')

## 8. Method agreement (A vs B)

In [ ]:
def jaccard(a, b):
    sa, sb = set(a), set(b)
    if not sa and not sb:
        return 1.0
    return len(sa & sb) / len(sa | sb)

jaccard_AB = pd.DataFrame(
    {
        lang: [jaccard(top_features_A[layer][lang], top_features_B[layer][lang])
               for layer in QWEN_SAE_SUBSET_LAYERS]
        for lang in TARGET_LANGUAGES
    },
    index=[f'L{l}' for l in QWEN_SAE_SUBSET_LAYERS],
)
print('Jaccard overlap (top-50, A vs B):')
print(jaccard_AB.round(3))

intersection_features = {
    layer: {
        lang: sorted(set(top_features_A[layer][lang]) & set(top_features_B[layer][lang]))
        for lang in TARGET_LANGUAGES
    }
    for layer in QWEN_SAE_SUBSET_LAYERS
}
print('\nIntersection sizes per (layer, language):')
for layer in QWEN_SAE_SUBSET_LAYERS:
    sizes = {l: len(intersection_features[layer][l]) for l in TARGET_LANGUAGES}
    print(f'  layer {layer:>2}: {sizes}')

## 9. Phase 2b — Causal ablation (H1: removing language features improves reasoning)

Ablate top intersection features at the best middle layer and evaluate reasoning accuracy.

In [ ]:
from src.evaluation import extract_numeric_answer, is_correct

# Use layer 11 (middle layer) for causal ablation
ABLATION_LAYER = 11
N_EVAL = 50  # prompts per language for generation eval

# Get feature directions for ablation
ablation_results = {}

for lang in TARGET_LANGUAGES:
    if lang == 'en':
        continue  # English is the reference — no language features to ablate

    feat_indices = intersection_features[ABLATION_LAYER].get(lang, [])
    if not feat_indices:
        feat_indices = top_features_A[ABLATION_LAYER][lang][:10]
    else:
        feat_indices = feat_indices[:10]  # top-10 for ablation

    directions = get_sae_decoder_directions(
        saes[ABLATION_LAYER], feat_indices, sae_type='batchtopk'
    )

    ablation_config = {ABLATION_LAYER: directions}

    # Baseline (no ablation)
    eval_prompts = prompts_by_lang[lang][:N_EVAL]
    eval_answers = [ex['answer_number'] for ex in mgsm[lang][:N_EVAL]]

    baseline_outputs = run_generate_with_hooks(
        model, tokenizer, eval_prompts,
        ablation_config={},  # no ablation
        max_new_tokens=512,
    )

    # Ablated
    ablated_outputs = run_generate_with_hooks(
        model, tokenizer, eval_prompts,
        ablation_config=ablation_config,
        max_new_tokens=512,
    )

    # Score
    baseline_correct = sum(
        is_correct(extract_numeric_answer(out), ans)
        for out, ans in zip(baseline_outputs, eval_answers)
    )
    ablated_correct = sum(
        is_correct(extract_numeric_answer(out), ans)
        for out, ans in zip(ablated_outputs, eval_answers)
    )

    ablation_results[lang] = {
        'baseline_acc': baseline_correct / N_EVAL,
        'ablated_acc': ablated_correct / N_EVAL,
        'delta': (ablated_correct - baseline_correct) / N_EVAL,
        'n_features_ablated': len(feat_indices),
    }
    print(f'{lang}: baseline={baseline_correct}/{N_EVAL} '
          f'ablated={ablated_correct}/{N_EVAL} '
          f'delta={ablation_results[lang]["delta"]:+.2f}')

    torch.cuda.empty_cache()

## 10. Cross-model comparison figures

In [ ]:
FIG_DIR = Path('results/figures'); FIG_DIR.mkdir(exist_ok=True, parents=True)
sns.set_theme(style='whitegrid', context='paper')

# Load Gemma results for comparison
gemma_phase1_path = RESULTS_DIR / 'phase1_features.pt'
gemma_phase2_path = RESULTS_DIR / 'phase2_ablation.pt'

has_gemma = gemma_phase1_path.exists()
if has_gemma:
    gemma_p1 = torch.load(gemma_phase1_path, weights_only=False)
    print('Loaded Gemma Phase 1 results for comparison')
else:
    print('No Gemma results found — showing Qwen-only figures')

In [ ]:
# Figure: Probe accuracy comparison (Qwen vs Gemma)
fig, ax = plt.subplots(figsize=(7, 3.5))

x = np.arange(len(QWEN_SAE_SUBSET_LAYERS))
width = 0.35

qwen_accs = [probe_accuracies[l] for l in QWEN_SAE_SUBSET_LAYERS]
ax.bar(x, qwen_accs, width, label='Qwen2.5-7B', color='#e8833a')

if has_gemma:
    gemma_layers = gemma_p1['config']['layers']
    gemma_accs = [gemma_p1['probe_accuracies'][l] for l in gemma_layers]
    x2 = np.arange(len(gemma_layers))
    ax.bar(x2 + width, gemma_accs, width, label='Gemma 2 9B', color='#5a86b5')

ax.set_ylim(0, 1.05)
ax.set_ylabel('Probe accuracy')
ax.set_xlabel('Layer (relative depth)')
ax.set_xticks(x)
ax.set_xticklabels([f'~{l/QWEN_N_LAYERS*100:.0f}%' for l in QWEN_SAE_SUBSET_LAYERS])
ax.legend()
ax.set_title('Language separability in SAE features (probe accuracy)')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig_cross_model_probe.png', dpi=150)
plt.show()

In [ ]:
# Figure: Ablation delta comparison
if ablation_results:
    fig, ax = plt.subplots(figsize=(6, 3.5))
    langs = [l for l in TARGET_LANGUAGES if l != 'en']
    deltas = [ablation_results[l]['delta'] for l in langs]
    ax.bar(langs, deltas, color=['#e8833a' if d > 0 else '#c44e52' for d in deltas])
    ax.axhline(0, color='k', linewidth=0.5)
    ax.set_ylabel('Accuracy delta (ablated - baseline)')
    ax.set_xlabel('Language')
    ax.set_title(f'H1: Effect of removing language features (Qwen, layer {ABLATION_LAYER})')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig_cross_model_ablation.png', dpi=150)
    plt.show()

## 11. Save Qwen results

In [ ]:
qwen_payload = {
    'config': {
        'model_id': QWEN_MODEL_ID,
        'sae_repo': 'andyrdt/saes-qwen2.5-7b-instruct',
        'layers': QWEN_SAE_SUBSET_LAYERS,
        'sae_width': QWEN_SAE_WIDTH,
        'trainer': QWEN_SAE_TRAINER,
        'languages': TARGET_LANGUAGES,
        'top_k': TOP_K,
    },
    'monolinguality': {
        layer: {lang: monolinguality_per_layer[layer][lang]
                for lang in TARGET_LANGUAGES}
        for layer in QWEN_SAE_SUBSET_LAYERS
    },
    'top_features_A': top_features_A,
    'top_features_B': top_features_B,
    'intersection_features': intersection_features,
    'reasoning_features': reasoning_features_per_layer,
    'probe_accuracies': probe_accuracies,
    'jaccard_AB': jaccard_AB.to_dict(),
    'ablation_results': ablation_results,
}

out_path = RESULTS_DIR / 'qwen_cross_model.pt'
torch.save(qwen_payload, out_path)
size_mb = out_path.stat().st_size / 1e6
print(f'Saved {out_path} ({size_mb:.1f} MB)')

if DRIVE_RESULTS is not None:
    drive_path = DRIVE_RESULTS / 'qwen_cross_model.pt'
    torch.save(qwen_payload, drive_path)
    print(f'Also saved to {drive_path}')

In [ ]:
# Final summary
print('=' * 60)
print('QWEN CROSS-MODEL SUMMARY')
print('=' * 60)
print(f'\nProbe accuracies:')
for layer in QWEN_SAE_SUBSET_LAYERS:
    print(f'  layer {layer:>2}: {probe_accuracies[layer]:.3f}')
print(f'\nMethod A\u2229B intersection sizes:')
for layer in QWEN_SAE_SUBSET_LAYERS:
    sizes = {l: len(intersection_features[layer][l]) for l in TARGET_LANGUAGES}
    print(f'  layer {layer:>2}: {sizes}')
print(f'\nReasoning-candidate counts:')
for layer in QWEN_SAE_SUBSET_LAYERS:
    print(f'  layer {layer:>2}: {len(reasoning_features_per_layer[layer])} features')
print(f'\nAblation results (H1):')
for lang, res in ablation_results.items():
    print(f'  {lang}: baseline={res["baseline_acc"]:.2f} '
          f'ablated={res["ablated_acc"]:.2f} delta={res["delta"]:+.3f}')